<a href="https://colab.research.google.com/github/sebaacosta2708/Robo-11/blob/main/Copia_de_Robt_de_inversi%C3%B3n_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ================================================================
# ROBOT DE INVERSIONES 11.0 - ALPACA
# AUTOMATIZACION DE COMPRAS Y VENTAS
# GOOGLE COLAB - UNA SOLA CELDA
# MODO SEGURO: PAPER TRADING
# ================================================================
# Estrategia: cruce EMA 9/21 + filtro RSI
# Protecciones: capital por posicion, limite de posiciones,
# stop-loss, take-profit, control de ordenes y registro CSV.
#
# IMPORTANTE:
# - PAPER=True impide que este codigo use dinero real.
# - No cambies el modo hasta terminar las pruebas.
# - No hay garantia de beneficios.
# ================================================================

!pip -q install --upgrade alpaca-py pandas

import os
import csv
import time
import uuid
import logging
import getpass
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, GetOrdersRequest
from alpaca.trading.enums import OrderSide, TimeInForce, QueryOrderStatus
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# ================================================================
# 1. CONFIGURACION PRINCIPAL
# ================================================================

VERSION = "11.0"

# BLOQUEO DE SEGURIDAD: solo simulacion.
PAPER = True
MODO_AUTOMATICO = True

# Acciones estadounidenses.
ACTIVOS = ["AAPL", "MSFT", "NVDA"]

# Limites de riesgo.
MAX_POSICIONES = 3
MAX_USD_POR_COMPRA = 25.00
MAX_PORCENTAJE_CAPITAL_POR_ACTIVO = 0.10
MAX_ORDENES_DIARIAS = 6

# Salidas automaticas.
STOP_LOSS_PCT = 0.03       # 3% por debajo del precio medio
TAKE_PROFIT_PCT = 0.05     # 5% por encima del precio medio

# Estrategia.
EMA_RAPIDA = 9
EMA_LENTA = 21
RSI_PERIODO = 14
RSI_MIN_COMPRA = 40
RSI_MAX_COMPRA = 68

# El ciclo se repite cada 60 segundos.
INTERVALO_SEGUNDOS = 60

# Archivos de auditoria.
CARPETA = Path("/content/robot_inversiones_11")
CARPETA.mkdir(parents=True, exist_ok=True)

ARCHIVO_OPERACIONES = CARPETA / "operaciones.csv"
ARCHIVO_ESTADO = CARPETA / "estado_robot.csv"
ARCHIVO_ERRORES = CARPETA / "errores.log"

logging.basicConfig(
    filename=str(ARCHIVO_ERRORES),
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

print("=" * 65)
print("ROBOT DE INVERSIONES", VERSION)
print("Modo: PAPER TRADING - DINERO SIMULADO")
print("Compras y ventas automaticas:", MODO_AUTOMATICO)
print("Activos:", ", ".join(ACTIVOS))
print("=" * 65)

# ================================================================
# 2. CREDENCIALES SEGURAS
# ================================================================

API_KEY = getpass.getpass("Introduce tu PAPER API KEY: ").strip()
API_SECRET = getpass.getpass("Introduce tu PAPER SECRET KEY: ").strip()

if not API_KEY or not API_SECRET:
    raise ValueError("Debes introducir las dos claves de PAPER TRADING.")

# Las claves no se guardan en los archivos CSV.
trading = TradingClient(API_KEY, API_SECRET, paper=True)
market_data = StockHistoricalDataClient(API_KEY, API_SECRET)

# ================================================================
# 3. ARCHIVOS Y REGISTROS
# ================================================================

CAMPOS_OPERACIONES = [
    "fecha_utc", "version", "activo", "accion",
    "cantidad", "precio_referencia", "importe_referencia",
    "motivo", "modo", "id_orden", "estado"
]

CAMPOS_ESTADO = [
    "fecha_utc", "activo", "precio", "ema_rapida",
    "ema_lenta", "rsi", "senal", "detalle"
]

def preparar_csv(ruta, campos):
    if not ruta.exists():
        with open(ruta, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=campos).writeheader()

preparar_csv(ARCHIVO_OPERACIONES, CAMPOS_OPERACIONES)
preparar_csv(ARCHIVO_ESTADO, CAMPOS_ESTADO)

def registrar_operacion(
    activo, accion, cantidad, precio, motivo,
    id_orden="", estado="REGISTRADO"
):
    importe = float(cantidad) * float(precio)
    fila = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "version": VERSION,
        "activo": activo,
        "accion": accion,
        "cantidad": round(float(cantidad), 8),
        "precio_referencia": round(float(precio), 4),
        "importe_referencia": round(importe, 2),
        "motivo": motivo,
        "modo": "PAPER",
        "id_orden": str(id_orden),
        "estado": estado
    }
    with open(ARCHIVO_OPERACIONES, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CAMPOS_OPERACIONES).writerow(fila)

def registrar_estado(activo, precio, ema9, ema21, rsi, senal, detalle):
    fila = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(),
        "activo": activo,
        "precio": round(float(precio), 4),
        "ema_rapida": round(float(ema9), 4),
        "ema_lenta": round(float(ema21), 4),
        "rsi": round(float(rsi), 2),
        "senal": senal,
        "detalle": detalle
    }
    with open(ARCHIVO_ESTADO, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CAMPOS_ESTADO).writerow(fila)

# ================================================================
# 4. CONEXION Y VERIFICACION DE CUENTA
# ================================================================

cuenta = trading.get_account()

if str(cuenta.status).lower().split(".")[-1] != "active":
    raise RuntimeError(f"La cuenta no esta activa: {cuenta.status}")

if getattr(cuenta, "trading_blocked", False):
    raise RuntimeError("Alpaca informa que las operaciones estan bloqueadas.")

print("\nConexion con Alpaca correcta.")
print("Cuenta:", cuenta.status)
print("Equity simulado: USD", cuenta.equity)
print("Efectivo disponible: USD", cuenta.cash)
print("Modo PAPER confirmado.")

# ================================================================
# 5. INDICADORES Y DATOS DEL MERCADO
# ================================================================

def obtener_datos(activo):
    ahora = datetime.now(timezone.utc)

    solicitud = StockBarsRequest(
        symbol_or_symbols=activo,
        timeframe=TimeFrame.Minute,
        start=ahora - timedelta(days=5),
        end=ahora - timedelta(minutes=1),
        limit=500
    )

    respuesta = market_data.get_stock_bars(solicitud)
    df = respuesta.df

    if df is None or df.empty:
        return None

    # Alpaca suele devolver un indice con activo y timestamp.
    if isinstance(df.index, pd.MultiIndex):
        try:
            df = df.xs(activo, level="symbol")
        except Exception:
            return None

    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")].copy()

    if "close" not in df.columns or len(df) < 50:
        return None

    df["ema_rapida"] = df["close"].ewm(
        span=EMA_RAPIDA, adjust=False
    ).mean()

    df["ema_lenta"] = df["close"].ewm(
        span=EMA_LENTA, adjust=False
    ).mean()

    cambio = df["close"].diff()
    ganancias = cambio.clip(lower=0)
    perdidas = -cambio.clip(upper=0)

    media_ganancias = ganancias.ewm(
        alpha=1 / RSI_PERIODO, min_periods=RSI_PERIODO,
        adjust=False
    ).mean()

    media_perdidas = perdidas.ewm(
        alpha=1 / RSI_PERIODO, min_periods=RSI_PERIODO,
        adjust=False
    ).mean()

    rs = media_ganancias / media_perdidas.replace(0, float("nan"))
    df["rsi"] = 100 - (100 / (1 + rs))
    df["rsi"] = df["rsi"].fillna(50)

    return df.dropna(subset=["ema_rapida", "ema_lenta"])

def obtener_posiciones():
    posiciones = {}
    for p in trading.get_all_positions():
        posiciones[p.symbol] = p
    return posiciones

def obtener_ordenes_abiertas():
    solicitud = GetOrdersRequest(
        status=QueryOrderStatus.OPEN,
        limit=500
    )
    return trading.get_orders(filter=solicitud)

def orden_pendiente(activo, ordenes):
    return any(o.symbol == activo for o in ordenes)

# ================================================================
# 6. CONTROL DEL LIMITE DE OPERACIONES DIARIAS
# ================================================================

fecha_contador = datetime.now(timezone.utc).date()
ordenes_hoy = 0

def actualizar_dia():
    global fecha_contador, ordenes_hoy
    hoy = datetime.now(timezone.utc).date()
    if hoy != fecha_contador:
        fecha_contador = hoy
        ordenes_hoy = 0

# ================================================================
# 7. ENVIO CONTROLADO DE ORDENES
# ================================================================

def enviar_compra(activo, importe, precio, motivo):
    global ordenes_hoy

    if not MODO_AUTOMATICO or not PAPER:
        print("Bloqueo de seguridad: compra no autorizada.")
        return

    if importe <= 1:
        return

    solicitud = MarketOrderRequest(
        symbol=activo,
        notional=round(float(importe), 2),
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY,
        client_order_id="R11B" + uuid.uuid4().hex[:20]
    )

    orden = trading.submit_order(order_data=solicitud)
    ordenes_hoy += 1

    registrar_operacion(
        activo, "COMPRA", importe / precio, precio, motivo,
        orden.id, str(orden.status)
    )

    print(
        f"COMPRA ENVIADA | {activo} | "
        f"USD {importe:.2f} | Orden {orden.id}"
    )

def enviar_venta(activo, posicion, precio, motivo):
    global ordenes_hoy

    if not MODO_AUTOMATICO or not PAPER:
        print("Bloqueo de seguridad: venta no autorizada.")
        return

    cantidad = abs(float(posicion.qty))
    if cantidad <= 0:
        return

    solicitud = MarketOrderRequest(
        symbol=activo,
        qty=round(cantidad, 6),
        side=OrderSide.SELL,
        time_in_force=TimeInForce.DAY,
        client_order_id="R11S" + uuid.uuid4().hex[:20]
    )

    orden = trading.submit_order(order_data=solicitud)
    ordenes_hoy += 1

    registrar_operacion(
        activo, "VENTA", cantidad, precio, motivo,
        orden.id, str(orden.status)
    )

    print(
        f"VENTA ENVIADA | {activo} | "
        f"Cantidad {cantidad:.6f} | Orden {orden.id}"
    )

# ================================================================
# 8. CICLO DE DECISION AUTOMATICA
# ================================================================

ultimas_velas_procesadas = {}

def ejecutar_ciclo():
    global ordenes_hoy

    actualizar_dia()

    reloj = trading.get_clock()
    if not reloj.is_open:
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            "Mercado cerrado. Esperando apertura."
        )
        return

    cuenta_actual = trading.get_account()
    equity = float(cuenta_actual.equity)
    buying_power = float(cuenta_actual.buying_power)

    posiciones = obtener_posiciones()
    ordenes = obtener_ordenes_abiertas()

    print(
        f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] "
        f"Equity USD {equity:.2f} | "
        f"Posiciones {len(posiciones)} | "
        f"Ordenes hoy {ordenes_hoy}"
    )

    for activo in ACTIVOS:
        try:
            df = obtener_datos(activo)

            if df is None or len(df) < 30:
                print(activo, "- Datos insuficientes.")
                continue

            actual = df.iloc[-1]
            anterior = df.iloc[-2]
            precio = float(actual["close"])
            ema9 = float(actual["ema_rapida"])
            ema21 = float(actual["ema_lenta"])
            rsi = float(actual["rsi"])
            hora_vela = str(df.index[-1])

            # No tomar decisiones dos veces sobre la misma vela.
            if ultimas_velas_procesadas.get(activo) == hora_vela:
                continue

            ultimas_velas_procesadas[activo] = hora_vela

            ema9_anterior = float(anterior["ema_rapida"])
            ema21_anterior = float(anterior["ema_lenta"])

            cruce_alcista = (
                ema9_anterior <= ema21_anterior
                and ema9 > ema21
            )

            cruce_bajista = (
                ema9_anterior >= ema21_anterior
                and ema9 < ema21
            )

            senal = "MANTENER"
            detalle = "Sin señal nueva"

            # Primero se controlan las posiciones existentes.
            posicion = posiciones.get(activo)

            if orden_pendiente(activo, ordenes):
                senal = "ESPERAR"
                detalle = "Hay una orden abierta; no duplicar."

            elif posicion is not None:
                # Este robot solo gestiona posiciones largas.
                # Si existe una posicion corta, no la modifica.
                cantidad_posicion = float(posicion.qty)

                if cantidad_posicion < 0:
                    senal = "BLOQUEADO"
                    detalle = "Posicion corta preexistente; revision manual."

                else:
                    precio_entrada = float(posicion.avg_entry_price)
                    rendimiento = (precio / precio_entrada) - 1

                    if rendimiento <= -STOP_LOSS_PCT:
                        senal = "VENDER"
                        detalle = "Stop-loss alcanzado"
                        enviar_venta(activo, posicion, precio, detalle)

                    elif rendimiento >= TAKE_PROFIT_PCT:
                        senal = "VENDER"
                        detalle = "Objetivo de beneficio alcanzado"
                        enviar_venta(activo, posicion, precio, detalle)

                    elif cruce_bajista:
                        senal = "VENDER"
                        detalle = "Cruce bajista de medias"
                        enviar_venta(activo, posicion, precio, detalle)

                    else:
                        senal = "MANTENER"
                        detalle = f"Posicion abierta; variacion {rendimiento:.2%}"

            # Si no hay posicion, buscar una compra.
            elif cruce_alcista and RSI_MIN_COMPRA <= rsi <= RSI_MAX_COMPRA:

                if len(posiciones) >= MAX_POSICIONES:
                    senal = "NO COMPRAR"
                    detalle = "Limite de posiciones alcanzado."

                elif ordenes_hoy >= MAX_ORDENES_DIARIAS:
                    senal = "NO COMPRAR"
                    detalle = "Limite diario de ordenes alcanzado."

                else:
                    # Presupuesto limitado por capital y saldo disponible.
                    limite_capital = max(
                        0.0, equity * MAX_PORCENTAJE_CAPITAL_POR_ACTIVO
                    )
                    importe = min(
                        MAX_USD_POR_COMPRA,
                        limite_capital,
                        buying_power * 0.95
                    )

                    if importe >= 2:
                        senal = "COMPRAR"
                        detalle = "Cruce alcista y filtro RSI valido"
                        enviar_compra(activo, importe, precio, detalle)
                        # Evita reutilizar el mismo buying power en el ciclo.
                        buying_power = max(0, buying_power - importe)
                    else:
                        senal = "NO COMPRAR"
                        detalle = "Capital disponible insuficiente."

            elif cruce_bajista:
                senal = "MANTENER"
                detalle = "Cruce bajista sin posicion abierta."

            registrar_estado(
                activo, precio, ema9, ema21, rsi, senal, detalle
            )

            print(
                f"{activo}: precio={precio:.2f} | "
                f"EMA{EMA_RAPIDA}={ema9:.2f} | "
                f"EMA{EMA_LENTA}={ema21:.2f} | "
                f"RSI={rsi:.1f} | {senal} | {detalle}"
            )

        except Exception as e:
            logging.exception("Error procesando %s", activo)
            print(f"ERROR en {activo}: {type(e).__name__}: {e}")

# ================================================================
# 9. EJECUCION CONTINUA
# ================================================================

print("\nRobot iniciado.")
print("Se revisara el mercado automaticamente.")
print("Para detenerlo, pulsa el boton de detener de Google Colab.")
print("Los archivos se guardan en:", CARPETA)
print("No cierres la sesion si quieres que siga funcionando.")

try:
    while True:
        try:
            ejecutar_ciclo()
        except Exception as e:
            logging.exception("Error general del ciclo")
            print("Error general; se reintentara:", type(e).__name__, str(e))

        time.sleep(INTERVALO_SEGUNDOS)

except KeyboardInterrupt:
    print("\nRobot detenido por el usuario.")

finally:
    print("Ejecucion finalizada.")
    print("Registro de operaciones:", ARCHIVO_OPERACIONES)
    print("Registro de señales:", ARCHIVO_ESTADO)
    print("Registro de errores:", ARCHIVO_ERRORES)


ROBOT DE INVERSIONES 11.0
Modo: PAPER TRADING - DINERO SIMULADO
Compras y ventas automaticas: True
Activos: AAPL, MSFT, NVDA

Conexion con Alpaca correcta.
Cuenta: AccountStatus.ACTIVE
Equity simulado: USD 100000.05
Efectivo disponible: USD 99900.01
Modo PAPER confirmado.

Robot iniciado.
Se revisara el mercado automaticamente.
Para detenerlo, pulsa el boton de detener de Google Colab.
Los archivos se guardan en: /content/robot_inversiones_11
No cierres la sesion si quieres que siga funcionando.
[20:03:13] Mercado cerrado. Esperando apertura.
[20:04:13] Mercado cerrado. Esperando apertura.
[20:05:14] Mercado cerrado. Esperando apertura.
[20:06:14] Mercado cerrado. Esperando apertura.
[20:07:14] Mercado cerrado. Esperando apertura.
[20:08:14] Mercado cerrado. Esperando apertura.
[20:09:14] Mercado cerrado. Esperando apertura.
[20:10:14] Mercado cerrado. Esperando apertura.
[20:11:14] Mercado cerrado. Esperando apertura.
[20:12:14] Mercado cerrado. Esperando apertura.
[20:13:14] Mercado 